# Findings and Review Method | Issue #11

Maintain a versioned findings register, methodology approvals, and independent review records for Gate 2.

Each finding must link to specific evidence and separate an observed result from its interpretation and possible October implications.

The notebook preserves existing register entries when rerun. Pending reviews and approvals must be completed by the team; generating a record does not count as approval.

In [1]:
from pathlib import Path
import hashlib
import re

import pandas as pd


def find_repo_root():
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError(
        "Repository root not found. Run this notebook inside the cloned repo."
    )


ROOT = find_repo_root()

GATE2 = (
    ROOT / "notebooks" / "final-deliverables"
    / "September" / "Gate 2"
)

assert GATE2.is_dir(), f"Gate 2 folder not found: {GATE2}"

FINDINGS_PATH = GATE2 / "findings_register.csv"
APPROVALS_PATH = GATE2 / "method_approvals.csv"
REVIEWS_PATH = GATE2 / "independent_review_register.csv"

REGISTER_VERSION = "1.0.0"


def relative_path(path):
    return path.resolve().relative_to(ROOT.resolve()).as_posix()


def artifact_sha256(path):
    """Identify the exact contents of an artifact, including uncommitted edits."""
    path = Path(path)

    if not path.is_file():
        return ""

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def load_or_create(path, rows, columns):
    """Create a register once; preserve manual edits on subsequent runs."""
    if path.exists():
        result = pd.read_csv(path, dtype=str, keep_default_na=False)

        missing_columns = set(columns) - set(result.columns)

        if missing_columns:
            raise ValueError(
                f"{path.name} is missing columns: {sorted(missing_columns)}"
            )

        print(f"Loaded existing {path.name}. No entries overwritten.")

        return result[columns].copy()

    result = pd.DataFrame(rows, columns=columns)
    result.to_csv(path, index=False)

    print(f"Created {path.name}.")

    return result


print("Repository:", ROOT)
print("Gate 2:", relative_path(GATE2))
print("Register version:", REGISTER_VERSION)

Repository: C:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity
Gate 2: notebooks/final-deliverables/September/Gate 2
Register version: 1.0.0


## 1. Identify the Final Evidence

Point each entry to the actual artifact in this branch.

Artifact paths are repository-relative. The notebook checks that each referenced file exists and records its current SHA-256 hash.

Do not mark an artifact approved merely because its file exists.

In [4]:
ARTIFACTS = {
    "data_dictionary": (
        "notebooks/final-deliverables/September/Gate 2/"
        "data_dictionary.csv"
    ),
    "target_analysis": (
        "notebooks/final-deliverables/September/Gate 2/"
        "target_analysis.ipynb"
    ),
    "categorical_analysis": (
        "notebooks/final-deliverables/September/Gate 2/"
        "categorical_analysis_method.ipynb"
    ),
    "continuous_analysis": (
        "notebooks/liam-stapley/"
        "continuous_analysis.ipynb"
    ),
    "bin_inventory": (
        "notebooks/final-deliverables/September/Gate 2/"
        "continuous_analysis_evidence/bin_inventory.csv"
    ),
}

artifact_rows = []

for artifact_name, rel_path in ARTIFACTS.items():
    path = ROOT / rel_path

    artifact_rows.append({
        "artifact": artifact_name,
        "path": rel_path,
        "exists": path.is_file(),
        "sha256": artifact_sha256(path),
    })

artifact_inventory = pd.DataFrame(artifact_rows)

display(artifact_inventory)

missing = artifact_inventory.loc[
    ~artifact_inventory["exists"], "path"
].tolist()

if missing:
    raise FileNotFoundError(
        "Update ARTIFACTS to match your actual files:\n"
        + "\n".join(missing)
    )

print("All listed evidence artifacts found.")

,artifact,path,exists,sha256
0,data_dictionary,notebooks/final-deliverables/September/Gate 2/...,True,ff63a03ace0bee921a17b9535ca1e55cde62d8b7b987f6...
1,target_analysis,notebooks/final-deliverables/September/Gate 2/...,True,18b2b4fc9926a41be137abf1f5329c8f867db3f82610a4...
2,categorical_analysis,notebooks/final-deliverables/September/Gate 2/...,True,98ac02eb9c2fb13e36bb06e9439ae92ff9416931de79d0...
3,continuous_analysis,notebooks/liam-stapley/continuous_analysis.ipynb,True,dfd2d8d959e60ff9109ddb1e45715852b3398f3db9da35...
4,bin_inventory,notebooks/final-deliverables/September/Gate 2/...,True,d6e02276a07f252a8439af158c05a508e52e0be1610cb7...


All listed evidence artifacts found.


## 2. Findings Register

A finding is a documented observation, not a claim about causation or model performance.

Every entry requires a stable finding ID, specific evidence location, observed result, interpretation, evidence status, limitation or alternative explanation, possible October implication, owner, and reviewer.

Use the project's four evidence-status terms:

- directly observed
- consistent with a pattern
- hypothesis
- inconclusive

The starter register contains proposed findings for the team to review. Replace generic evidence locations with the specific table row, field, plot, or notebook section supporting each entry.

In [5]:
FINDING_COLUMNS = [
    "finding_id",
    "title",
    "evidence_artifact",
    "evidence_path",
    "evidence_locator",
    "evidence_sha256",
    "observed_result",
    "interpretation",
    "evidence_status",
    "limitation_or_alternative_explanation",
    "possible_october_implication",
    "owner",
    "reviewer",
    "review_decision",
    "review_notes",
    "register_version",
]

VALID_EVIDENCE_STATUSES = {
    "directly observed",
    "consistent with a pattern",
    "hypothesis",
    "inconclusive",
}

# This is an initial entry based on the target-analysis results reviewed
# for Issue #10. Confirm the precise evidence locator before approval.
starter_findings = [
    {
        "finding_id": "F-001",
        "title": "Raw loss has a long upper tail",
        "evidence_artifact": "target_analysis",
        "evidence_path": ARTIFACTS["target_analysis"],
        "evidence_locator": (
            "Target summary table and raw loss histogram; "
            "confirm exact notebook section/cell"
        ),
        "evidence_sha256": artifact_sha256(
            ROOT / ARTIFACTS["target_analysis"]
        ),
        "observed_result": (
            "The saved target summary reports mean loss of $3,037.34, "
            "median loss of $2,115.57, and positive skewness of 3.795."
        ),
        "interpretation": (
            "The observed loss distribution is right-skewed, "
            "with larger claims contributing to the upper tail."
        ),
        "evidence_status": "directly observed",
        "limitation_or_alternative_explanation": (
            "The raw histogram limits its displayed range for readability. "
            "The full-distribution statistics and excluded share must "
            "be considered alongside the plot."
        ),
        "possible_october_implication": (
            "Examine model errors across the loss distribution, "
            "including high-cost claims. Retain MAE in original loss units."
        ),
        "owner": "Mia Chavez / Junaid Pathan",
        "reviewer": "Liam Stapley",
        "review_decision": "pending",
        "review_notes": (
            "Confirm the evidence locator and final target-analysis "
            "artifact after Issue #10 review changes."
        ),
        "register_version": REGISTER_VERSION,
    }
]

findings = load_or_create(
    FINDINGS_PATH,
    starter_findings,
    FINDING_COLUMNS,
)

display(findings)

Created findings_register.csv.


,finding_id,title,evidence_artifact,evidence_path,evidence_locator,evidence_sha256,observed_result,interpretation,evidence_status,limitation_or_alternative_explanation,possible_october_implication,owner,reviewer,review_decision,review_notes,register_version
0,F-001,Raw loss has a long upper tail,target_analysis,notebooks/final-deliverables/September/Gate 2/...,Target summary table and raw loss histogram; c...,18b2b4fc9926a41be137abf1f5329c8f867db3f82610a4...,The saved target summary reports mean loss of ...,The observed loss distribution is right-skewed...,directly observed,The raw histogram limits its displayed range f...,Examine model errors across the loss distribut...,Mia Chavez / Junaid Pathan,Liam Stapley,pending,Confirm the evidence locator and final target-...,1.0.0


## 3. Team Method Approvals

Record the team's decisions on the six methods required by Issue #11.

An approval must identify the method, reviewed artifact, exact artifact version, approver, date, and any disagreement or requested change.

Leave decisions as pending until the team actually confirms them.

In [6]:
APPROVAL_COLUMNS = [
    "method",
    "artifact",
    "artifact_path",
    "artifact_sha256",
    "decision",
    "approved_by",
    "decision_date",
    "disagreements_or_required_changes",
    "notes",
]

method_artifacts = [
    ("Data dictionary", "data_dictionary"),
    ("Target views", "target_analysis"),
    ("Rare-level rule", "categorical_analysis"),
    ("Continuous-bin rule", "bin_inventory"),
    ("Correlation methods", "continuous_analysis"),
]

starter_approvals = []

for method, artifact in method_artifacts:
    path = ROOT / ARTIFACTS[artifact]

    starter_approvals.append({
        "method": method,
        "artifact": artifact,
        "artifact_path": ARTIFACTS[artifact],
        "artifact_sha256": artifact_sha256(path),
        "decision": "pending",
        "approved_by": "",
        "decision_date": "",
        "disagreements_or_required_changes": "",
        "notes": "",
    })

# The findings-register format is the sixth required approval.
starter_approvals.append({
    "method": "Findings-register format",
    "artifact": "findings_register",
    "artifact_path": relative_path(FINDINGS_PATH),
    "artifact_sha256": artifact_sha256(FINDINGS_PATH),
    "decision": "pending",
    "approved_by": "",
    "decision_date": "",
    "disagreements_or_required_changes": "",
    "notes": "",
})

approvals = load_or_create(
    APPROVALS_PATH,
    starter_approvals,
    APPROVAL_COLUMNS,
)

display(approvals)

Created method_approvals.csv.


,method,artifact,artifact_path,artifact_sha256,decision,approved_by,decision_date,disagreements_or_required_changes,notes
0,Data dictionary,data_dictionary,notebooks/final-deliverables/September/Gate 2/...,ff63a03ace0bee921a17b9535ca1e55cde62d8b7b987f6...,pending,,,,
1,Target views,target_analysis,notebooks/final-deliverables/September/Gate 2/...,18b2b4fc9926a41be137abf1f5329c8f867db3f82610a4...,pending,,,,
2,Rare-level rule,categorical_analysis,notebooks/final-deliverables/September/Gate 2/...,98ac02eb9c2fb13e36bb06e9439ae92ff9416931de79d0...,pending,,,,
3,Continuous-bin rule,bin_inventory,notebooks/final-deliverables/September/Gate 2/...,d6e02276a07f252a8439af158c05a508e52e0be1610cb7...,pending,,,,
4,Correlation methods,continuous_analysis,notebooks/liam-stapley/continuous_analysis.ipynb,dfd2d8d959e60ff9109ddb1e45715852b3398f3db9da35...,pending,,,,
5,Findings-register format,findings_register,notebooks/final-deliverables/September/Gate 2/...,ad503980844e6ead8b27204a845fa538cfc36804ccf007...,pending,,,,


## 4. Independent Review Register

Record the independent reviewer and exact artifact version for each Gate 2 deliverable.

The reviewer should not be the person who produced the artifact being reviewed.

Reviews already in progress may be recorded with their actual status. Do not mark them approved until the requested changes and final artifact verification are complete.

In [7]:
REVIEW_COLUMNS = [
    "issue",
    "artifact",
    "artifact_path",
    "artifact_sha256",
    "author",
    "independent_reviewer",
    "review_date",
    "review_decision",
    "fresh_run_verified",
    "disagreements",
    "limitations",
    "review_notes",
]

review_items = [
    (
        "#10", "target_analysis",
        "Mia Chavez / Junaid Pathan", "Liam Stapley"
    ),
    (
        "#15", "data_dictionary",
        "Liam Stapley", ""
    ),
    (
        "#16", "categorical_analysis",
        "@ezixuan27 / @jonathandeng7", "Liam Stapley"
    ),
    (
        "#17", "continuous_analysis",
        "Liam Stapley / Ragib Nehal", ""
    ),
]

starter_reviews = []

for issue, artifact, author, reviewer in review_items:
    path = ROOT / ARTIFACTS[artifact]

    starter_reviews.append({
        "issue": issue,
        "artifact": artifact,
        "artifact_path": ARTIFACTS[artifact],
        "artifact_sha256": artifact_sha256(path),
        "author": author,
        "independent_reviewer": reviewer,
        "review_date": "",
        "review_decision": "pending",
        "fresh_run_verified": "pending",
        "disagreements": "",
        "limitations": "",
        "review_notes": "",
    })

starter_reviews.append({
    "issue": "#11",
    "artifact": "findings_register",
    "artifact_path": relative_path(FINDINGS_PATH),
    "artifact_sha256": artifact_sha256(FINDINGS_PATH),
    "author": "Allstate 1A team",
    "independent_reviewer": "",
    "review_date": "",
    "review_decision": "pending",
    "fresh_run_verified": "not applicable",
    "disagreements": "",
    "limitations": "",
    "review_notes": "",
})

reviews = load_or_create(
    REVIEWS_PATH,
    starter_reviews,
    REVIEW_COLUMNS,
)

display(reviews)

Created independent_review_register.csv.


,issue,artifact,artifact_path,artifact_sha256,author,independent_reviewer,review_date,review_decision,fresh_run_verified,disagreements,limitations,review_notes
0,#10,target_analysis,notebooks/final-deliverables/September/Gate 2/...,18b2b4fc9926a41be137abf1f5329c8f867db3f82610a4...,Mia Chavez / Junaid Pathan,Liam Stapley,,pending,pending,,,
1,#15,data_dictionary,notebooks/final-deliverables/September/Gate 2/...,ff63a03ace0bee921a17b9535ca1e55cde62d8b7b987f6...,Liam Stapley,,,pending,pending,,,
2,#16,categorical_analysis,notebooks/final-deliverables/September/Gate 2/...,98ac02eb9c2fb13e36bb06e9439ae92ff9416931de79d0...,@ezixuan27 / @jonathandeng7,Liam Stapley,,pending,pending,,,
3,#17,continuous_analysis,notebooks/liam-stapley/continuous_analysis.ipynb,dfd2d8d959e60ff9109ddb1e45715852b3398f3db9da35...,Liam Stapley / Ragib Nehal,,,pending,pending,,,
4,#11,findings_register,notebooks/final-deliverables/September/Gate 2/...,ad503980844e6ead8b27204a845fa538cfc36804ccf007...,Allstate 1A team,,,pending,not applicable,,,


## 5. Validate Documentation and Report Readiness

Check the completeness of the findings register, methodology approvals, and independent reviews.

A valid CSV does not automatically mean Gate 2 has passed. Final conclusions remain pending until all required approvals and reviews are recorded against the final artifact versions.

In [8]:
findings = pd.read_csv(
    FINDINGS_PATH, dtype=str, keep_default_na=False
)

approvals = pd.read_csv(
    APPROVALS_PATH, dtype=str, keep_default_na=False
)

reviews = pd.read_csv(
    REVIEWS_PATH, dtype=str, keep_default_na=False
)

# Findings structure
assert findings["finding_id"].is_unique
assert findings["finding_id"].str.fullmatch(r"F-\d{3,}").all()

assert findings["evidence_status"].isin(
    VALID_EVIDENCE_STATUSES
).all()

required_finding_fields = [
    "finding_id",
    "title",
    "evidence_path",
    "evidence_locator",
    "observed_result",
    "interpretation",
    "evidence_status",
    "limitation_or_alternative_explanation",
    "possible_october_implication",
    "owner",
    "reviewer",
    "register_version",
]

assert findings[required_finding_fields].ne("").all().all(), (
    "One or more findings are missing required documentation."
)

# Evidence must actually exist on this branch.
for _, finding in findings.iterrows():
    path = ROOT / finding["evidence_path"]

    assert path.is_file(), (
        f"Missing evidence for {finding['finding_id']}: {path}"
    )

# Method approvals
required_methods = {
    "Data dictionary",
    "Target views",
    "Rare-level rule",
    "Continuous-bin rule",
    "Correlation methods",
    "Findings-register format",
}

assert set(approvals["method"]) == required_methods
assert approvals["method"].is_unique

assert approvals["decision"].isin({
    "pending", "approved", "changes requested"
}).all()

# Independent reviews
assert reviews["review_decision"].isin({
    "pending", "approved", "changes requested"
}).all()

# Check for stale approvals/reviews if artifacts have changed.
stale_approvals = []

for _, row in approvals.iterrows():
    path = ROOT / row["artifact_path"]

    if (
        row["decision"] == "approved"
        and row["artifact_sha256"] != artifact_sha256(path)
    ):
        stale_approvals.append(row["method"])

stale_reviews = []

for _, row in reviews.iterrows():
    path = ROOT / row["artifact_path"]

    if (
        row["review_decision"] == "approved"
        and row["artifact_sha256"] != artifact_sha256(path)
    ):
        stale_reviews.append(row["issue"])

finding_reviews_complete = (
    findings["review_decision"].eq("approved").all()
    and findings["reviewer"].ne("").all()
)

method_approvals_complete = (
    approvals["decision"].eq("approved").all()
    and approvals[["approved_by", "decision_date"]].ne("").all().all()
    and not stale_approvals
)

independent_reviews_complete = (
    reviews["review_decision"].eq("approved").all()
    and reviews[
        ["independent_reviewer", "review_date"]
    ].ne("").all().all()
    and not stale_reviews
)

print("Findings recorded:", len(findings))
print("Finding reviews complete:", finding_reviews_complete)
print("Method approvals complete:", method_approvals_complete)
print("Independent reviews complete:", independent_reviews_complete)
print("Approvals for changed artifacts:", stale_approvals)
print("Reviews for changed artifacts:", stale_reviews)

if (
    finding_reviews_complete
    and method_approvals_complete
    and independent_reviews_complete
):
    print("\nDocumentation status: READY FOR FINAL GATE 2 SIGN-OFF")
else:
    print("\nDocumentation status: PENDING")
    print("Complete the remaining findings, approvals, and reviews.")

Findings recorded: 1
Finding reviews complete: False
Method approvals complete: False
Independent reviews complete: False
Approvals for changed artifacts: []
Reviews for changed artifacts: []

Documentation status: PENDING
Complete the remaining findings, approvals, and reviews.
